# Attribution Tear Sheet

**Use after source analytics:** Reporting notebooks render results produced by the pricing, analytics, statement, and portfolio notebooks; they do not replace those calculation workflows.

**Purpose:** Render return or P&L attribution results into a compact review page.

**Prerequisites:** `03_analytics/return_contribution.ipynb` and `02_pricing/pnl_attribution.ipynb`.

**What you'll learn:**

- Prepare attribution result inputs.
- Render `reporting.attribution_tearsheet`.
- Explain contribution results without recomputing attribution in the reporting layer.

A **P&L attribution tear sheet** decomposes an instrument's profit-and-loss between two
dates (T₀ → T₁) into the factors that drove it: **carry** (coupon income, roll-down,
theta), **rates**, **credit**, **FX**, **vol**, and a residual. The headline is a
**contribution waterfall** that bridges from zero through each factor to the total P&L,
backed by a factor table and a carry / roll-down breakdown.

`reporting.attribution_tearsheet(attribution, ...)` is a pure formatter over a
`PnlAttribution` (produced by `attribution.attribute_pnl`). Like the other sheets, the
returned `TearSheet` renders inline in Jupyter via `_repr_html_` — just make it the last
expression in a cell. You can also pass `instrument=` plus two markets to have it compute
the attribution for you.


In [5]:
import datetime as dt
import json
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.attribution import PnlAttribution, attribute_pnl
from finstack_quant import reporting

In [6]:
# A USD fixed-rate bond: ACME 4.25% maturing 2034-03-15.
bond = json.dumps({
    "type": "bond", 
    "spec": {
        "id": "ACME 4.25% 2034",
        "notional": {"amount": "10000000", "currency": "USD"},
        "issue_date": "2024-03-15",
        "maturity": "2034-03-15",
        "cashflow_spec": {"Fixed": {
            "coupon_type": "Cash", 
            "rate": 0.0425,
            "freq": {"count": 6, "unit": "months"},
            "dc": "Thirty360", 
            "bdc": "following",
            "calendar_id": "weekends_only", 
            "stub": "None",
            "end_of_month": False, 
            "payment_lag_days": 0}
        },
        "call_put": None,    
        "attributes": {
            "tags": [], 
            "meta": {},    
        },
        "discount_curve_id": "USD-OIS",
        "pricing_overrides": {}
        }
})


# Two market snapshots one month apart on the same USD-OIS curve.
# Subtracting from the discount factors at T1 lifts implied rates (a modest sell-off).
def usd_ois(as_of: str, shift: float = 0.0) -> str:
    mc = MarketContext()
    mc.insert(DiscountCurve(
        "USD-OIS", date.fromisoformat(as_of),
        [(0.0, 1.0), (0.5, 0.98 - shift), (1.0, 0.96 - shift), (2.0, 0.92 - shift),
         (3.0, 0.88 - shift), (5.0, 0.80 - shift), (10.0, 0.65 - shift)],
        day_count="act_365f"))
    return mc.to_json()


t0, t1 = "2025-01-15", "2025-02-18"
market_t0 = usd_ois(t0)
market_t1 = usd_ois(t1, shift=0.004)
print(f"Attribution window: {t0} → {t1}")

Attribution window: 2025-01-15 → 2025-02-18


In [7]:
# Decompose the P&L between the two dates with a parallel (full-reprice) attribution.
attribution = PnlAttribution.from_json(
    attribute_pnl(bond, market_t0, market_t1, t0, t1, "Parallel"))

cur = attribution.currency
print(f"Total P&L:  {attribution.total_pnl:>12,.2f} {cur}")
print(f"  Carry:    {attribution.carry:>12,.2f}")
print(f"  Rates:    {attribution.rates_curves_pnl:>12,.2f}")
print(f"  Residual: {attribution.residual:>12,.2f}")
print(f"Repricings: {attribution.num_repricings}")

Total P&L:    -17,555.91 USD
  Carry:       37,497.25
  Rates:      -55,053.15
  Residual:         0.00
Repricings: 6


In [8]:
# Full attribution tear sheet — rendered inline via _repr_html_.
# The contribution waterfall bridges from 0 through each factor to Total P&L; below it,
# a factor table and the carry / roll-down breakdown (the deferred instrument-sheet detail).
reporting.attribution_tearsheet(attribution, generated=dt.date(2026, 6, 21))

TearSheet(theme=Theme(name='institutional', font_head='Georgia,"Times New Roman",serif', font_num='"Iowan Old Style",Georgia,serif', font_sans="-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif", ink='#10243f', muted='#7a8190', pos='#1b6b4f', neg='#9b2335', accent='#7c2230', canvas='#fbfaf6', rule='#10243f', faint='#e4e1d8', grid='#cfc8b8'), title='P&L Attribution', sections=[Section(title='P&L Attribution', body='<svg viewBox="0 0 620 210" xmlns="http://www.w3.org/2000/svg"><line x1="56" y1="170.0" x2="606" y2="170.0" stroke="#e4e1d8"/><text x="50" y="173.0" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">-20,000</text><line x1="56" y1="143.7" x2="606" y2="143.7" stroke="#e4e1d8"/><text x="50" y="146.7" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">-10,000</text><line x1="56" y1="117.3" x2="606" y2="117.3" stroke="#cfc8b8"/><text x="50" y="120.3" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">0</text><line x1="56" y1="91.0" x2="606" y2="91.0" stroke="#e4e1d8"/><text x="50" y="94.0" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">10,000</text><line x1="56" y1="64.7" x2="606" y2="64.7" stroke="#e4e1d8"/><text x="50" y="67.7" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">20,000</text><line x1="56" y1="38.3" x2="606" y2="38.3" stroke="#e4e1d8"/><text x="50" y="41.3" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">30,000</text><line x1="56" y1="12.0" x2="606" y2="12.0" stroke="#e4e1d8"/><text x="50" y="15.0" text-anchor="end" font-size="10" fill="#7a8190" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">40,000</text><rect class="fq-hb" x="90.8" y="18.6" width="113.7" height="98.7" fill="#1b6b4f" fill-opacity="0.82" data-cx="147.7" data-cy="18.6" data-label="Carry" data-val="+37,497"><title>Carry · +37,497</title></rect><text x="147.7" y="184" text-anchor="middle" font-size="9.5" fill="#7a8190" font-family="-apple-system,BlinkMacSystemFont,\'Segoe UI\',Roboto,sans-serif">Carry</text><text x="147.7" y="14.6" text-anchor="middle" font-size="9.5" fill="#23303f" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">+37,497</text><line x1="204.5" y1="18.6" x2="274.2" y2="18.6" stroke="#e4e1d8" stroke-dasharray="2 2"/><rect class="fq-hb" x="274.2" y="18.6" width="113.7" height="145.0" fill="#9b2335" fill-opacity="0.82" data-cx="331.0" data-cy="18.6" data-label="Rates" data-val="−55,053"><title>Rates · −55,053</title></rect><text x="331.0" y="184" text-anchor="middle" font-size="9.5" fill="#7a8190" font-family="-apple-system,BlinkMacSystemFont,\'Segoe UI\',Roboto,sans-serif">Rates</text><text x="331.0" y="175.6" text-anchor="middle" font-size="9.5" fill="#23303f" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">−55,053</text><line x1="387.8" y1="163.6" x2="457.5" y2="163.6" stroke="#e4e1d8" stroke-dasharray="2 2"/><rect class="fq-hb" x="457.5" y="117.3" width="113.7" height="46.2" fill="#10243f" fill-opacity="0.9" data-cx="514.3" data-cy="117.3" data-label="Total P&amp;L" data-val="−17,556"><title>Total P&L · −17,556</title></rect><text x="514.3" y="184" text-anchor="middle" font-size="9.5" fill="#7a8190" font-family="-apple-system,BlinkMacSystemFont,\'Segoe UI\',Roboto,sans-serif">Total P&L</text><text x="514.3" y="175.6" text-anchor="middle" font-size="9.5" fill="#23303f" font-family="&quot;Iowan Old Style&quot;,Georgia,serif">−17,556</text><line class="fq-cross" x1="0" x2="0" y1="12" y2="170" style="visibility:hidden" pointer-events="none"/><circle class="fq-mk" r="3.5" cx="0" cy="0" style="visibility:hidden" pointer-events="none"/></svg>', subtitle=None), Section(title='Factor Contributions', body='<table class="dd"><thead><tr><th>Factor</th><th>Amount</th><th>% of Total</

## Takeaways

- Reporting functions are presentation wrappers over analytics, valuation, statement, or portfolio results produced earlier in the curriculum.
- Keep the analytical source of truth in the typed objects or JSON specs, then render a tear sheet for review.
- Pass fixed `generated` dates in examples so notebook output remains reproducible.
